# Cat vs Dog Image Classification - Experiment Notebook

This notebook is the submission for Exercise 1. It trains and compares several image-classification approaches for classifying input images as `cat` or `dog`, then selects the best method by validation F1 score and accuracy.

Target runtime: Kaggle Notebook or Google Colab with GPU.

## Solution Summary

- Dataset: public Kaggle Dogs vs Cats dataset.
- Framework: PyTorch.
- Models tested: ResNet18, MobileNetV3-Small, EfficientNet-B0, ViT-Small, ViT-Small partial fine-tuning, ViT-Small with LoRA, and the best CNN with stronger augmentation.
- Selection rule: choose the best validation F1 score; use validation accuracy as the tie-breaker.
- Output: a saved checkpoint at `/content/models/best_model.pt` and a prediction function for one image file.

## 1. Install Dependencies

Run this cell first. In Kaggle Notebooks, most dependencies are already available, but this keeps the environment consistent. Kaggle credentials are only needed if you use the optional Colab download cell below.

In [1]:
!pip -q install timm scikit-learn pandas tqdm kaggle

import os
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

import timm
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

## 2. Configuration and Reproducibility

In [2]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 512
NUM_WORKERS = 2
MAX_IMAGES_PER_CLASS = 3000  # set to None for the full dataset
SMOKE_TEST = False            # set True to validate the pipeline quickly

KAGGLE_ENV = Path('/kaggle/input').exists()
ROOT = Path('/kaggle/working') if KAGGLE_ENV else Path('/content')
DATA_ROOT = ROOT / 'data'
RAW_DIR = DATA_ROOT / 'raw'
def find_petimages_root():
    candidates = []
    if KAGGLE_ENV:
        candidates.extend(Path('/kaggle/input').glob('**/PetImages'))
    candidates.extend([
        Path('/content/Cats and Dogs Classification Dataset/PetImages'),
        ROOT / 'Cats and Dogs Classification Dataset' / 'PetImages',
    ])
    for path in candidates:
        if (path / 'Cat').exists() and (path / 'Dog').exists():
            return path
        if (path / 'cat').exists() and (path / 'dog').exists():
            return path
    return candidates[0] if candidates else Path('/kaggle/input')

# Default Kaggle dataset layout:
# /kaggle/input/<dataset-name>/PetImages/Cat
# /kaggle/input/<dataset-name>/PetImages/Dog
IMAGE_ROOT = find_petimages_root()

# Optional converted Kaggle layout used only if you call prepare_dogs_vs_cats().
KAGGLE_IMAGE_ROOT = DATA_ROOT / 'cats_dogs'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()
print('Seed:', SEED)
print('Root:', ROOT)
print('Image root:', IMAGE_ROOT)

Seed: 42
Root: /kaggle/working
Image root: /kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages


## 3. Dataset Setup

Preferred Kaggle Notebook path: add the dataset to the notebook from the Kaggle UI. The notebook auto-detects this layout:

```text
/kaggle/input/<dataset-name>/PetImages/
  Cat/
    *.jpg
  Dog/
    *.jpg
```

This is auto-detected as `IMAGE_ROOT`. In Colab, either place the folder at `/content/Cats and Dogs Classification Dataset/PetImages` or use the optional Kaggle API download/preparation cells.

In [3]:
# Optional: download from Kaggle in Colab.
# 1. In Colab, upload kaggle.json when prompted.
# 2. This cell downloads the Dogs vs Cats dataset.

USE_KAGGLE_DOWNLOAD = False

if USE_KAGGLE_DOWNLOAD:
    from google.colab import files
    uploaded = files.upload()
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    kaggle_json = kaggle_dir / 'kaggle.json'
    if not kaggle_json.exists():
        Path('kaggle.json').rename(kaggle_json)
    os.chmod(kaggle_json, 0o600)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c dogs-vs-cats -p /content/data/raw
    print('Downloaded files:', list(RAW_DIR.glob('*')))

In [4]:
def prepare_dogs_vs_cats(raw_dir=RAW_DIR, image_root=KAGGLE_IMAGE_ROOT, max_images_per_class=MAX_IMAGES_PER_CLASS):
    image_root = Path(image_root)
    cat_dir = image_root / 'cat'
    dog_dir = image_root / 'dog'
    if cat_dir.exists() and dog_dir.exists() and len(list(cat_dir.glob('*'))) > 0 and len(list(dog_dir.glob('*'))) > 0:
        print('Prepared dataset already exists:', image_root)
        return image_root

    raw_dir = Path(raw_dir)
    image_root.mkdir(parents=True, exist_ok=True)
    cat_dir.mkdir(exist_ok=True)
    dog_dir.mkdir(exist_ok=True)

    # Kaggle competition download contains dogs-vs-cats.zip, which contains train.zip.
    outer_zip = raw_dir / 'dogs-vs-cats.zip'
    train_zip = raw_dir / 'train.zip'
    if outer_zip.exists() and not train_zip.exists():
        with zipfile.ZipFile(outer_zip, 'r') as zf:
            zf.extractall(raw_dir)

    train_extract = raw_dir / 'train'
    if train_zip.exists() and not train_extract.exists():
        with zipfile.ZipFile(train_zip, 'r') as zf:
            zf.extractall(raw_dir)

    source_images = sorted((raw_dir / 'train').glob('*.jpg'))
    if not source_images:
        raise FileNotFoundError('No training images found. Enable USE_KAGGLE_DOWNLOAD or provide /content/data/raw/train/*.jpg')

    counts = {'cat': 0, 'dog': 0}
    for src in tqdm(source_images, desc='Preparing folders'):
        label = src.name.split('.')[0]
        if label not in counts:
            continue
        if max_images_per_class is not None and counts[label] >= max_images_per_class:
            continue
        dst = image_root / label / src.name
        if not dst.exists():
            dst.write_bytes(src.read_bytes())
        counts[label] += 1

    print('Prepared counts:', counts)
    return image_root

# Run after downloading or manually placing Kaggle images in /content/data/raw/train.
# This creates /content/data/cats_dogs/cat and /content/data/cats_dogs/dog.
# If you use this path instead of PetImages, set IMAGE_ROOT = KAGGLE_IMAGE_ROOT after running it.
# prepare_dogs_vs_cats()
# IMAGE_ROOT = KAGGLE_IMAGE_ROOT

## 4. Data Loaders and Transforms

In [5]:
def get_transforms(strong_aug=False):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if strong_aug:
        train_tfms = transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        train_tfms = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize,
        ])

    val_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tfms, val_tfms

def normalize_class_names(dataset):
    normalized = {name.lower(): idx for name, idx in dataset.class_to_idx.items()}
    if 'cat' not in normalized or 'dog' not in normalized:
        raise ValueError(f"Expected class folders named Cat/Dog or cat/dog, found: {dataset.classes}")
    return {'cat': normalized['cat'], 'dog': normalized['dog']}

def build_loaders(image_root=IMAGE_ROOT, strong_aug=False, smoke_test=SMOKE_TEST):
    train_tfms, val_tfms = get_transforms(strong_aug=strong_aug)
    base = datasets.ImageFolder(image_root)
    labels = [y for _, y in base.samples]
    indices = list(range(len(base)))
    train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=SEED, stratify=labels)

    if smoke_test:
        train_idx = train_idx[:128]
        val_idx = val_idx[:64]

    train_ds = datasets.ImageFolder(image_root, transform=train_tfms)
    val_ds = datasets.ImageFolder(image_root, transform=val_tfms)

    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(Subset(val_ds, val_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    class_to_idx = normalize_class_names(base)
    print('Classes:', base.classes, base.class_to_idx)
    print('Train images:', len(train_idx), 'Validation images:', len(val_idx))
    return train_loader, val_loader, class_to_idx


## 5. Model Factory

LoRA is implemented locally for the ViT `qkv` attention projection layers. For this small assignment, this avoids adding a complicated external adapter framework while still using parameter-efficient fine-tuning.

In [6]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError('LoRALinear expects nn.Linear')
        self.base = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)
        self.lora_a = nn.Parameter(torch.zeros(rank, base_layer.in_features))
        self.lora_b = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_a, a=np.sqrt(5))
        nn.init.zeros_(self.lora_b)
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        base_out = self.base(x)
        lora_weight = self.lora_b @ self.lora_a
        return base_out + nn.functional.linear(self.dropout(x), lora_weight) * self.scale

def apply_lora_to_vit_qkv(model, rank=8, alpha=16, dropout=0.05):
    replaced = 0
    for block in getattr(model, 'blocks', []):
        attn = getattr(block, 'attn', None)
        if attn is not None and isinstance(getattr(attn, 'qkv', None), nn.Linear):
            attn.qkv = LoRALinear(attn.qkv, rank=rank, alpha=alpha, dropout=dropout)
            replaced += 1
    if replaced == 0:
        raise RuntimeError('No ViT qkv layers were replaced with LoRA')
    return model

def set_trainable(model, mode):
    for p in model.parameters():
        p.requires_grad = False

    if mode == 'head':
        modules = []
        if hasattr(model, 'classifier'):
            modules.append(model.classifier)
        if hasattr(model, 'fc'):
            modules.append(model.fc)
        if hasattr(model, 'head'):
            modules.append(model.head)
        for module in modules:
            for p in module.parameters():
                p.requires_grad = True
    elif mode == 'partial_vit':
        for p in model.head.parameters():
            p.requires_grad = True
        for block in model.blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True
    elif mode == 'lora':
        for name, p in model.named_parameters():
            if 'lora_' in name or name.startswith('head'):
                p.requires_grad = True
    elif mode == 'full':
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f'Unknown trainable mode: {mode}')

def create_model(model_name, trainable_mode='head', use_lora=False):
    if model_name == 'resnet18':
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif model_name == 'mobilenet_v3_small':
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_small(weights=weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    elif model_name == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    elif model_name == 'vit_small_patch16_224':
        model = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=2)
        if use_lora:
            model = apply_lora_to_vit_qkv(model)
    else:
        raise ValueError(f'Unknown model: {model_name}')

    set_trainable(model, trainable_mode)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'{model_name}: trainable {trainable_params:,} / total {total_params:,}')
    return model.to(DEVICE)

## 6. Training and Evaluation

In [7]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, leave=False, desc='train'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    y_true, y_pred = [], []
    for images, labels in tqdm(loader, leave=False, desc='val'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss = criterion(logits, labels)
        preds = logits.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
    return {
        'val_loss': running_loss / len(loader.dataset),
        'val_acc': accuracy_score(y_true, y_pred),
        'val_f1': f1_score(y_true, y_pred, average='macro'),
    }

def run_experiment(config, train_loader, val_loader):
    seed_everything()
    start = time.time()
    model = create_model(config['model_name'], config['trainable_mode'], config.get('use_lora', False))
    criterion = nn.CrossEntropyLoss(label_smoothing=config.get('label_smoothing', 0.0))
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=config['lr'], weight_decay=config.get('weight_decay', 1e-4))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    best_metrics = None
    ckpt_path = MODEL_DIR / f"{config['name']}.pt"

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        metrics = evaluate(model, val_loader, criterion)
        scheduler.step()
        metrics['train_loss'] = train_loss
        metrics['epoch'] = epoch
        print(config['name'], 'epoch', epoch, metrics)
        if best_metrics is None or metrics['val_f1'] > best_metrics['val_f1']:
            best_metrics = metrics.copy()
            torch.save({'model_state': model.state_dict(), 'config': config, 'metrics': best_metrics}, ckpt_path)

    elapsed = time.time() - start
    result = {**config, **best_metrics, 'seconds': round(elapsed, 1), 'checkpoint': str(ckpt_path)}
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 7. Experiment List

Use `SMOKE_TEST = True` near the top to verify that the full pipeline works before running all experiments.

In [8]:
EXPERIMENTS = [
    {'name': 'resnet18_head', 'model_name': 'resnet18', 'trainable_mode': 'head', 'lr': 3e-4, 'epochs': 3},
    {'name': 'mobilenet_v3_small_head', 'model_name': 'mobilenet_v3_small', 'trainable_mode': 'head', 'lr': 3e-4, 'epochs': 3},
    {'name': 'efficientnet_b0_head', 'model_name': 'efficientnet_b0', 'trainable_mode': 'head', 'lr': 3e-4, 'epochs': 3},
    {'name': 'vit_small_head', 'model_name': 'vit_small_patch16_224', 'trainable_mode': 'head', 'lr': 3e-4, 'epochs': 3},
    {'name': 'vit_small_partial', 'model_name': 'vit_small_patch16_224', 'trainable_mode': 'partial_vit', 'lr': 1e-5, 'epochs': 3},
    {'name': 'vit_small_lora', 'model_name': 'vit_small_patch16_224', 'trainable_mode': 'lora', 'use_lora': True, 'lr': 1e-4, 'epochs': 3},
]

CNN_STRONG_AUG_TEMPLATE = {
    'name': 'best_cnn_strong_aug',
    'model_name': None,
    'trainable_mode': 'head',
    'lr': 2e-4,
    'epochs': 4,
    'label_smoothing': 0.05,
}

EXPERIMENTS

[{'name': 'resnet18_head',
  'model_name': 'resnet18',
  'trainable_mode': 'head',
  'lr': 0.0003,
  'epochs': 3},
 {'name': 'mobilenet_v3_small_head',
  'model_name': 'mobilenet_v3_small',
  'trainable_mode': 'head',
  'lr': 0.0003,
  'epochs': 3},
 {'name': 'efficientnet_b0_head',
  'model_name': 'efficientnet_b0',
  'trainable_mode': 'head',
  'lr': 0.0003,
  'epochs': 3},
 {'name': 'vit_small_head',
  'model_name': 'vit_small_patch16_224',
  'trainable_mode': 'head',
  'lr': 0.0003,
  'epochs': 3},
 {'name': 'vit_small_partial',
  'model_name': 'vit_small_patch16_224',
  'trainable_mode': 'partial_vit',
  'lr': 1e-05,
  'epochs': 3},
 {'name': 'vit_small_lora',
  'model_name': 'vit_small_patch16_224',
  'trainable_mode': 'lora',
  'use_lora': True,
  'lr': 0.0001,
  'epochs': 3}]

## 8. Run Experiments

In [9]:
# Ensure your data exists at IMAGE_ROOT before running this cell.
# Kaggle expected path: /kaggle/input/<dataset-name>/PetImages/Cat and Dog.
# If using the optional Kaggle preparation flow instead, run:
# prepare_dogs_vs_cats(); IMAGE_ROOT = KAGGLE_IMAGE_ROOT

train_loader, val_loader, class_to_idx = build_loaders(IMAGE_ROOT, strong_aug=False)

results = []
for cfg in EXPERIMENTS:
    results.append(run_experiment(cfg, train_loader, val_loader))

results_df = pd.DataFrame(results).sort_values(['val_f1', 'val_acc'], ascending=False)
results_df

Classes: ['Cat', 'Dog'] {'Cat': 0, 'Dog': 1}
Train images: 19998 Validation images: 5000
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s] 


resnet18: trainable 1,026 / total 11,177,538


train:   0%|          | 0/40 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a4c82f87920>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a4c82f87920>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

val:   0%|          | 0/10 [00:00<?, ?it/s]

vit_small_partial epoch 3 {'val_loss': 0.07109099161624909, 'val_acc': 0.983, 'val_f1': 0.98299970011471, 'train_loss': 0.0809116157442212, 'epoch': 3}
vit_small_patch16_224: trainable 148,226 / total 21,813,890


train:   0%|          | 0/40 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 592.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 548.81 MiB is free. Including non-PyTorch memory, this process has 13.99 GiB memory in use. Of the allocated memory 13.54 GiB is allocated by PyTorch, and 321.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [16]:
# Load and print experiment results from saved checkpoints.
# This cell does not train anything.

CHECKPOINT_NAMES = [
    'efficientnet_b0_head.pt',
    'mobilenet_v3_small_head.pt',
    'resnet18_head.pt',
    'vit_small_head.pt',
    'vit_small_partial.pt',
    'vit_small_lora.pt',
]

def infer_config_from_checkpoint_name(checkpoint_name):
    stem = Path(checkpoint_name).stem
    for cfg in EXPERIMENTS:
        if cfg['name'] == stem:
            return cfg.copy()
    raise ValueError(f'Cannot infer config for checkpoint: {checkpoint_name}')

def load_checkpoint_file(ckpt_path):
    try:
        return torch.load(ckpt_path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(ckpt_path, map_location='cpu')

def evaluate_checkpoint_if_needed(ckpt_path, checkpoint, config, val_loader):
    metrics = checkpoint.get('metrics') or {}
    required_metrics = {'val_loss', 'val_acc', 'val_f1'}
    if required_metrics.issubset(metrics.keys()):
        return metrics

    print('Metrics missing; evaluating checkpoint:', ckpt_path.name)
    model = create_model(config['model_name'], config['trainable_mode'], config.get('use_lora', False))
    model.load_state_dict(checkpoint['model_state'])
    criterion = nn.CrossEntropyLoss(label_smoothing=config.get('label_smoothing', 0.0))
    metrics = evaluate(model, val_loader, criterion)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics

def load_experiment_results_from_checkpoints(model_dir=MODEL_DIR, checkpoint_names=CHECKPOINT_NAMES, val_loader=None):
    loaded = []
    for checkpoint_name in checkpoint_names:
        ckpt_path = Path(model_dir) / checkpoint_name
        if not ckpt_path.exists():
            print('Missing checkpoint, skipping:', ckpt_path)
            continue
        checkpoint = load_checkpoint_file(ckpt_path)
        config = checkpoint.get('config') or infer_config_from_checkpoint_name(checkpoint_name)
        metrics = evaluate_checkpoint_if_needed(ckpt_path, checkpoint, config, val_loader) if val_loader is not None else checkpoint.get('metrics', {})
        loaded.append({
            **config,
            **metrics,
            'checkpoint': str(ckpt_path),
        })
    if not loaded:
        raise FileNotFoundError(f'No experiment checkpoints found in {model_dir}')
    df = pd.DataFrame(loaded)
    missing_columns = {'val_f1', 'val_acc'} - set(df.columns)
    if missing_columns:
        raise ValueError(f'Missing metrics {missing_columns}. Pass val_loader to evaluate checkpoints.')
    return df.sort_values(['val_f1', 'val_acc'], ascending=False)

if 'val_loader' not in globals():
    _, val_loader, class_to_idx = build_loaders(IMAGE_ROOT, strong_aug=False)

results_df = load_experiment_results_from_checkpoints(val_loader=val_loader)
results_df = results_df.drop(columns=['lr','name','epoch', 'val_acc', 'checkpoint', 'train_loss', 'val_loss'], errors='ignore')
results = results_df.to_dict('records')

results_df

Missing checkpoint, skipping: /kaggle/working/models/vit_small_lora.pt


,model_name,trainable_mode,epochs,val_f1
3,vit_small_patch16_224,head,3,0.986000
4,vit_small_patch16_224,partial_vit,3,0.983000
1,mobilenet_v3_small,head,3,0.963800
0,efficientnet_b0,head,3,0.950396
2,resnet18,head,3,0.947398


## Report: Pipeline, Algorithms, Limitations, and Improvements

### Pipeline

1. Download or provide the Kaggle Dogs vs Cats dataset.
2. Load images from the `PetImages/Cat` and `PetImages/Dog` folders using `ImageFolder`.
3. Split the images into train and validation sets using a deterministic stratified split.
4. Train several pretrained image models using transfer learning.
5. Evaluate each experiment using validation loss, accuracy, and macro F1 score.
6. Select the best model by validation F1 and save it as the final checkpoint.
7. Predict the class of a new image with the final checkpoint.

### Algorithms Used

- CNN transfer learning with ResNet18, MobileNetV3-Small, and EfficientNet-B0.
- Vision Transformer transfer learning using `vit_small_patch16_224` from `timm`.
- Parameter-efficient ViT fine-tuning using LoRA adapters on attention `qkv` projections.
- Cross-entropy loss, AdamW optimizer, cosine learning-rate schedule, and ImageNet normalization.

### Remaining Problems

- Training images from the internet may include noisy labels, duplicates, watermarks, unusual crops, or non-cat/dog objects.
- Private evaluation images may differ in lighting, camera angle, breed, age, or image quality.
- A small validation split may not perfectly represent the private test set.

### Ideas for Improvement

- Clean the dataset by removing corrupt, duplicate, and mislabeled images.
- Use k-fold cross-validation for more stable model selection.
- Tune learning rate, batch size, augmentation strength, and number of unfrozen layers.
- Ensemble the best CNN and ViT models if inference speed is not important.
- Calibrate confidence scores using validation data.